# Customer Churn Prediction

## Contexto
O cancelamento de assinaturas ocorre quando os clientes param de usar o serviço de uma empresa, para empresas com modelos de negócios baseados em assinaturas (telecomunicações, streaming, saas) o cancelamento impacta diretamente a receita.

## Objetivo
Usando Machine Learning, vamos prever quais clientes têm mais probabilidade de cancelar serviços e assim, permitir que a empresa tome medidas preventivas.

## Autoria
Wellington M. Santos
- **Linkedin**: [in/wellington-moreira-santos](https://www.linkedin.com/in/wellington-moreira-santos/)
- **Github**: [esscova](https://github.com/esscova)

## Fluxo deste notebook
1. Model training
2. Model evaluation
3. Model comparison
4. Business insights
5. Model export for deployment

## 1. DEPENDÊNCIAS E CONFIGURAÇÕES

In [2]:
# diretorio
from pathlib import Path

# data manipulation
import pandas as pd
import numpy as np

# dataviz
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Machine learning models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc

# save trained models
import joblib

## 2. CARREGANDO DATASET

In [4]:
# preprocessed
df = pd.read_parquet(Path.cwd().parent/'data/preprocessed.parquet')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgCharges,NewCustomer
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,14.925000,1
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,No,53.985714,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,36.050000,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,40.016304,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,50.550000,1


## 3. MODELING

In [5]:
# ENCODING QUALITATIVOS
encoders = {}

for col in df.columns:
    if df[col].dtype == "object":

        le = LabelEncoder()

        df[col] = le.fit_transform(df[col])

        encoders[col] = le

In [6]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgCharges,NewCustomer
0,0,0,1,0,1,0,1,0,0,2,...,0,0,0,1,2,29.85,29.85,0,14.925000,1
1,1,0,0,0,34,1,0,0,2,0,...,0,0,1,0,3,56.95,1889.50,0,53.985714,0
2,1,0,0,0,2,1,0,0,2,2,...,0,0,0,1,3,53.85,108.15,1,36.050000,1
3,1,0,0,0,45,0,1,0,2,0,...,0,0,1,0,0,42.30,1840.75,0,40.016304,0
4,0,0,0,0,2,1,0,1,0,0,...,0,0,0,1,2,70.70,151.65,1,50.550000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,2,0,2,0,...,2,2,1,1,3,84.80,1990.50,0,79.620000,0
7039,0,0,1,1,72,1,2,1,0,2,...,2,2,1,1,1,103.20,7362.90,0,100.861644,0
7040,0,0,1,1,11,0,1,0,2,0,...,0,0,0,1,2,29.60,346.45,0,28.870833,1
7041,1,1,1,0,4,1,2,1,0,0,...,0,0,0,1,3,74.40,306.60,1,61.320000,1


## 4. SPLIT FEATURES

In [7]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [8]:
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,AvgCharges,NewCustomer
0,0,0,1,0,1,0,1,0,0,2,...,0,0,0,0,1,2,29.85,29.85,14.925000,1
1,1,0,0,0,34,1,0,0,2,0,...,0,0,0,1,0,3,56.95,1889.50,53.985714,0
2,1,0,0,0,2,1,0,0,2,2,...,0,0,0,0,1,3,53.85,108.15,36.050000,1
3,1,0,0,0,45,0,1,0,2,0,...,2,0,0,1,0,0,42.30,1840.75,40.016304,0
4,0,0,0,0,2,1,0,1,0,0,...,0,0,0,0,1,2,70.70,151.65,50.550000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,2,0,2,0,...,2,2,2,1,1,3,84.80,1990.50,79.620000,0
7039,0,0,1,1,72,1,2,1,0,2,...,0,2,2,1,1,1,103.20,7362.90,100.861644,0
7040,0,0,1,1,11,0,1,0,2,0,...,0,0,0,0,1,2,29.60,346.45,28.870833,1
7041,1,1,1,0,4,1,2,1,0,0,...,0,0,0,0,1,3,74.40,306.60,61.320000,1


In [9]:
y

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [10]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
X_train.shape, y_train.shape

((5634, 21), (5634,))

In [14]:
X_test.shape, y_test.shape

((1409, 21), (1409,))

## 5. SCALING

In [16]:
# scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

C:\Users\wsant\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [17]:
X_train

array([[-1.02516569, -0.4377492 , -0.96957859, ..., -0.42134513,
        -0.01422481, -0.64217136],
       [-1.02516569, -0.4377492 , -0.96957859, ...,  1.25588791,
         0.49310492, -0.64217136],
       [ 0.97545208, -0.4377492 ,  1.03137591, ..., -1.00215117,
        -0.77974648,  1.55721675],
       ...,
       [ 0.97545208, -0.4377492 ,  1.03137591, ..., -0.87717627,
        -0.59527142, -0.64217136],
       [ 0.97545208,  2.28441306, -0.96957859, ..., -0.4817762 ,
         0.47837639, -0.64217136],
       [ 0.97545208, -0.4377492 , -0.96957859, ..., -0.8102886 ,
        -0.69838354, -0.64217136]])

## 6. BASELINE MODEL

In [18]:
# modelo que classifica com classe mais frequente
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print('Baseline accuracy: ', accuracy_score(y_test, baseline_pred) )

Baseline accuracy:  0.7352732434350603
